# SupplySight AI
# Enterprise Procurement Decision Intelligence Platform

## Notebook 02A : Enterprise Data Modeling

---

## Objective

The objective of this notebook is to design a preliminary enterprise data model for the procurement analytics platform.

The initial model is based on business understanding and will be validated against the actual transactional grain of the DataCo Supply Chain dataset before implementation.

This notebook documents the logical warehouse design and identifies candidate fact and dimension tables.

The data model will support:

- Procurement Analytics
- Supplier Performance Analysis
- Inventory Intelligence
- Cost Optimization
- Predictive Procurement
- Executive Dashboards

This notebook defines the logical architecture before any feature engineering or SQL implementation.

In [14]:
import pandas as pd

df_dataCo = pd.read_csv(
    "../Datasets/Raw Datasets/DataCoSupplyChainDataset.csv",
    encoding="latin1"
)

In [15]:
df_dataCo.head()

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,Customer Country,Customer Email,Customer Fname,Customer Id,Customer Lname,Customer Password,Customer Segment,Customer State,Customer Street,Customer Zipcode,Department Id,Department Name,Latitude,Longitude,Market,Order City,Order Country,Order Customer Id,order date (DateOrders),Order Id,Order Item Cardprod Id,Order Item Discount,Order Item Discount Rate,Order Item Id,Order Item Product Price,Order Item Profit Ratio,Order Item Quantity,Sales,Order Item Total,Order Profit Per Order,Order Region,Order State,Order Status,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Cally,20755,Holloway,XXXXXXXXX,Consumer,PR,5365 Noble Nectar Island,725.0,2,Fitness,18.251453,-66.037056,Pacific Asia,Bekasi,Indonesia,20755,1/31/2018 22:56,77202,1360,13.110000,0.04,180517,327.75,0.29,1,327.75,314.640015,91.250000,Southeast Asia,Java Occidental,COMPLETE,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Irene,19492,Luna,XXXXXXXXX,Consumer,PR,2679 Rustic Loop,725.0,2,Fitness,18.279451,-66.037064,Pacific Asia,Bikaner,India,19492,1/13/2018 12:27,75939,1360,16.389999,0.05,179254,327.75,-0.80,1,327.75,311.359985,-249.089996,South Asia,Rajastán,PENDING,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,EE. UU.,XXXXXXXXX,Gillian,19491,Maldonado,XXXXXXXXX,Consumer,CA,8510 Round Bear Gate,95125.0,2,Fitness,37.292233,-121.881279,Pacific Asia,Bikaner,India,19491,1/13/2018 12:06,75938,1360,18.030001,0.06,179253,327.75,-0.80,1,327.75,309.720001,-247.779999,South Asia,Rajastán,CLOSED,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,EE. UU.,XXXXXXXXX,Tana,19490,Tate,XXXXXXXXX,Home Office,CA,3200 Amber Bend,90027.0,2,Fitness,34.125946,-118.291016,Pacific Asia,Townsville,Australia,19490,1/13/2018 11:45,75937,1360,22.940001,0.07,179252,327.75,0.08,1,327.75,304.809998,22.860001,Oceania,Queensland,COMPLETE,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Orli,19489,Hendricks,XXXXXXXXX,Corporate,PR,8671 Iron Anchor Corners,725.0,2,Fitness,18.253769,-66.037048,Pacific Asia,Townsville,Australia,19489,1/13/2018 11:24,75936,1360,29.500000,0.09,179251,327.75,0.45,1,327.75,298.250000,134.210007,Oceania,Queensland,PENDING_PAYMENT,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


### Observations

- The DataCo dataset operates at an **order-item transactional grain**, with each row representing an individual order item.
- The dataset contains attributes that can be logically separated into **fact measures, dimensions, identifiers, and operational fields**.
- Candidate dimensions include **Customer, Product, Category, Department, Market, Geography, Date, and Shipment**.
- Candidate fact tables will capture **sales, quantity, discount, profit, and delivery performance** measures.
- External datasets can be integrated as supporting **date-, geography-, and market-level dimensions/facts**.
- The proposed enterprise model will be validated against the actual transactional grain before physical implementation.

# Determine the Transaction Grain

Before designing the star schema, it is essential to determine the grain of the transactional dataset.

The grain defines what a single row represents.

Possible grains include:

- One Purchase Order
- One Order Item
- One Shipment
- One Inventory Transaction

The warehouse design will be based on the verified grain rather than assumptions.

In [16]:
print("Total Rows:", df_dataCo.shape[0])

print("Unique Orders:", df_dataCo["Order Id"].nunique())

print("Unique Order Items:", df_dataCo["Order Item Id"].nunique())

print("Unique Products:", df_dataCo["Product Card Id"].nunique())

print("Unique Customers:", df_dataCo["Customer Id"].nunique())

Total Rows: 180519
Unique Orders: 65752
Unique Order Items: 180519
Unique Products: 118
Unique Customers: 20652


In [17]:
df_dataCo.groupby("Order Id").size().sort_values(ascending=False).head(10)

Order Id
65699    5
42199    5
13260    5
42258    5
42246    5
42243    5
42236    5
13268    5
42222    5
42220    5
dtype: int64

### Observation

- **Grain**: Each record represents a single order item (order line) within a customer order. An order (Order Id) may contain multiple products, resulting in multiple records, while each record is uniquely identified by Order Item Id.
- The dataset contains **180,519 rows** and **180,519 unique Order Item IDs**.
- There are **65,752 unique Orders**, confirming that an order can contain multiple items.
- Each row therefore represents **one Order Item transaction** rather than one complete order.
- The verified transactional grain is: **One Order Item per row**.
- This grain will serve as the foundation for the **fact table and star schema design**.

# Enterprise Analytics Architecture

The platform follows a layered architecture.
```text
External APIs
↓
Raw Data
↓
Data Cleaning
↓
Enterprise Data Warehouse
↓
Feature Engineering
↓
Machine Learning
↓
Power BI Dashboard
```

In [18]:
import pandas as pd
import numpy as np

from pathlib import Path

pd.set_option("display.max_columns",None)

print("Libraries Imported Successfully")

Libraries Imported Successfully


# Load Core Datasets

In [19]:
PROJECT_PATH = Path(r"C:\Users\chavi\Desktop\SupplySight AI – Enterprise Procurement Decision Intelligence Platform")

RAW = PROJECT_PATH / "Datasets" / "Raw Datasets"

EXTERNAL = PROJECT_PATH / "Datasets" / "External Datasets"

In [20]:
dataco = pd.read_csv(
    RAW/"DataCoSupplyChainDataset.csv",
    encoding="latin1"
)

description = pd.read_csv(
    RAW/"DescriptionDataCoSupplyChain.csv",
    encoding="latin1"
)

fuel = pd.read_csv(
    EXTERNAL/"fuel_prices.csv"
)

holiday = pd.read_csv(
    EXTERNAL/"holidays_cleaned.csv"
)

# Procurement Business Process

The procurement lifecycle represented in this project consists of the following stages.

Supplier

↓

Purchase Order

↓

Inventory

↓

Shipment

↓

Customer Delivery

↓

Performance Evaluation

# Business Entities

The DataCo dataset represents multiple business entities instead of a single flat table.

The objective is to identify these entities and normalize them into an enterprise warehouse.

In [21]:
pd.DataFrame(
    dataco.columns,
    columns=["Available Columns"]
)

,Available Columns
0,Type
1,Days for shipping (real)
2,Days for shipment (scheduled)
3,Benefit per order
4,Sales per customer
5,Delivery Status
6,Late_delivery_risk
7,Category Id
8,Category Name
9,Customer City


### Observations

- The platform follows a **layered architecture** from raw data ingestion to analytics, machine learning, and dashboarding.
- The DataCo dataset combines multiple business entities within a single transactional table.
- Key entities identified include **Customer, Order, Product, Category, Department, Market, Geography, and Shipment/Delivery**.
- These entities provide the foundation for **normalization and star-schema development**.
- The architecture separates **data preparation, warehouse modeling, analytics, and ML layers**, supporting scalable decision intelligence.

# Entity Identification

The following entities have been identified.

1. Customer

2. Product

3. Order

4. Shipment

5. Supplier

6. Inventory

7. Category

8. Department

9. Market

10. Geography

These entities will later become fact and dimension tables.

# Candidate Fact Table

After analyzing the business process, the warehouse will be designed around a single central fact table.

The fact table represents the core business event occurring in the procurement lifecycle.

**Proposed Fact Table**

- Fact_Order_Items

Each row represents one product purchased within one customer order.

This grain will be validated using the DataCo Supply Chain dataset before final implementation.

In [22]:
fact_table = pd.DataFrame({

    "Fact Table": [
        "Fact_Order_Items"
    ],

    "Business Event": [
        "One product purchased within one customer order"
    ],

    "Grain": [
        "One Order Item"
    ]

})

fact_table

,Fact Table,Business Event,Grain
0,Fact_Order_Items,One product purchased within one customer order,One Order Item


### Observations

- The enterprise model identifies **Customer, Product, Order, Shipment, Supplier, Inventory, Category, Department, Market, and Geography** as key business entities.
- A central **Fact_Order_Items** table is proposed to capture the core transactional activity.
- The defined grain is **one product purchased within one customer order (one Order Item)**.
- This grain aligns with the verified DataCo transactional structure and provides a consistent basis for **KPI and dimensional analysis**.
- Supplier and Inventory are included in the target enterprise model but require **additional data sources** for implementation.

# Enterprise Dimension Tables

Dimension tables provide descriptive attributes that enrich the transactional data stored in the fact table.

These dimensions enable slicing, filtering, grouping, and drill-down analysis in SQL queries and Power BI dashboards.

The proposed dimensions will support procurement analytics, supplier performance analysis, inventory intelligence, and predictive modeling.

In [23]:
dimension_tables = pd.DataFrame({

    "Dimension Table":[
        "Dim_Date",
        "Dim_Product",
        "Dim_Customer",
        "Dim_Department",
        "Dim_Category",
        "Dim_Market",
        "Dim_Country",
        "Dim_Commodity",
        "Dim_Fuel",
        "Dim_ExchangeRate",
        "Dim_Weather",
        "Dim_Holiday"
    ],

    "Dimension Type":[
        "Internal",
        "Internal",
        "Internal",
        "Internal",
        "Internal",
        "Internal",
        "Reference",
        "External",
        "External",
        "External",
        "External",
        "External"
    ],

    "Business Purpose":[
        "Calendar hierarchy and time intelligence",
        "Product master information",
        "Customer demographics",
        "Department hierarchy",
        "Product category hierarchy",
        "Market segmentation",
        "Standardized country reference",
        "Commodity price trends",
        "Fuel price trends",
        "Currency exchange rates",
        "Country-level weather conditions",
        "Country-level public holiday calendar"
    ]

})

dimension_tables

,Dimension Table,Dimension Type,Business Purpose
0,Dim_Date,Internal,Calendar hierarchy and time intelligence
1,Dim_Product,Internal,Product master information
2,Dim_Customer,Internal,Customer demographics
3,Dim_Department,Internal,Department hierarchy
4,Dim_Category,Internal,Product category hierarchy
5,Dim_Market,Internal,Market segmentation
6,Dim_Country,Reference,Standardized country reference
7,Dim_Commodity,External,Commodity price trends
8,Dim_Fuel,External,Fuel price trends
9,Dim_ExchangeRate,External,Currency exchange rates


### Observations

- **12 dimension tables** are proposed to support descriptive and analytical requirements.
- Internal dimensions capture **date, product, customer, department, category, and market** attributes.
- **Dim_Country** provides standardized geographical reference data.
- External dimensions add **commodity, fuel, exchange-rate, weather, and holiday** context.
- These dimensions enable flexible **filtering, drill-down, KPI analysis, and feature engineering** across the platform.

### Proposed Grain

The preliminary assumption is that each row in the DataCo dataset represents one order item (one product within one customer order).

This assumption will be validated by examining the uniqueness of Order Id and Order Item Id.

Once validated, the fact table will be renamed accordingly.

# Primary Keys

In [24]:
primary_keys = pd.DataFrame({

    "Table":[
        "Fact_Order_Items",
        "Dim_Date",
        "Dim_Product",
        "Dim_Customer",
        "Dim_Department",
        "Dim_Category",
        "Dim_Market",
        "Dim_Country"

    ],

    "Primary Key":[
        "Order_Item_ID",
        "Date_Key",
        "Product_ID",
        "Customer_ID",
        "Department_ID",
        "Category_ID",
        "Market",
        "Country_Key"
    ]

})

primary_keys

,Table,Primary Key
0,Fact_Order_Items,Order_Item_ID
1,Dim_Date,Date_Key
2,Dim_Product,Product_ID
3,Dim_Customer,Customer_ID
4,Dim_Department,Department_ID
5,Dim_Category,Category_ID
6,Dim_Market,Market
7,Dim_Country,Country_Key


### Observations

- Primary keys have been defined for the **fact and core dimension tables**.
- `Order_Item_ID` uniquely identifies each transaction at the **order-item grain**.
- Dimension keys provide consistent references between the fact table and descriptive attributes.
- `Date_Key` and `Country_Key` support **time-based and geographical integration**.
- These keys establish the foundation for **fact–dimension relationships and star-schema implementation**.

# Enterprise Dimension Relationships

In [25]:
relationships = pd.DataFrame({

    "Fact Table":[
        "Fact_Order_Items",
        "Fact_Order_Items",
        "Fact_Order_Items",
        "Fact_Order_Items",
        "Fact_Order_Items",
        "Fact_Order_Items",
        "Fact_Order_Items",
        "Fact_Order_Items",
        "Fact_Order_Items",
        "Fact_Order_Items",
        "Fact_Order_Items"
    ],

    "Dimension":[
        "Dim_Date",
        "Dim_Product",
        "Dim_Customer",
        "Dim_Department",
        "Dim_Category",
        "Dim_Market",
        "Dim_Country",
        "Dim_Commodity",
        "Dim_Fuel",
        "Dim_ExchangeRate",
        "Dim_Weather / Dim_Holiday"

    ],

    "Join Key":[
        "Date_Key",
        "Product_ID",
        "Customer_ID",
        "Department_ID",
        "Category_ID",
        "Market",
        "Country_Key",
        "Year_Month",
        "Week_Start",
        "Transaction_Date",
        "Country + Date"
    ]

})

relationships

,Fact Table,Dimension,Join Key
0,Fact_Order_Items,Dim_Date,Date_Key
1,Fact_Order_Items,Dim_Product,Product_ID
2,Fact_Order_Items,Dim_Customer,Customer_ID
3,Fact_Order_Items,Dim_Department,Department_ID
4,Fact_Order_Items,Dim_Category,Category_ID
5,Fact_Order_Items,Dim_Market,Market
6,Fact_Order_Items,Dim_Country,Country_Key
7,Fact_Order_Items,Dim_Commodity,Year_Month
8,Fact_Order_Items,Dim_Fuel,Week_Start
9,Fact_Order_Items,Dim_ExchangeRate,Transaction_Date


### Observations

- `Fact_Order_Items` serves as the **central fact table** connected to multiple dimensions.
- Internal dimensions use stable business keys such as **Date, Product, Customer, Department, Category, Market, and Country**.
- External datasets are linked using **time- and geography-based keys** such as `Year_Month`, `Week_Start`, and `Country + Date`.
- These relationships enable integrated analysis of **sales, costs, exchange rates, weather, holidays, and operational performance**.
- The proposed joins will be **validated during physical implementation** to ensure correct grain and avoid duplicate matches.

# External Dataset Integration

In [26]:
external = pd.DataFrame({

    "Dataset":[
        "Commodity Prices",
        "Fuel Prices",
        "Exchange Rates",
        "Weather",
        "Holiday"
    ],

    "Warehouse Dimension":[
        "Dim_Commodity",
        "Dim_Fuel",
        "Dim_ExchangeRate",
        "Dim_Weather",
        "Dim_Holiday"
    ],

    "Granularity":[
        "Monthly",
        "Weekly",
        "Daily",
        "Country × Day",
        "Country × Day"

    ],

    "Join Key":[
        "Year_Month",
        "Week_Start",
        "Transaction_Date",
        "Country + Date",
        "Country + Date"
    ]

})

external

,Dataset,Warehouse Dimension,Granularity,Join Key
0,Commodity Prices,Dim_Commodity,Monthly,Year_Month
1,Fuel Prices,Dim_Fuel,Weekly,Week_Start
2,Exchange Rates,Dim_ExchangeRate,Daily,Transaction_Date
3,Weather,Dim_Weather,Country × Day,Country + Date
4,Holiday,Dim_Holiday,Country × Day,Country + Date


### Observations

- External datasets are integrated at their **native business granularity**: monthly, weekly, or daily.
- **Commodity and fuel prices** provide cost context at monthly and weekly levels.
- **Exchange rates** are integrated at the daily transaction-date level.
- **Weather and holiday data** are linked using **country + date** for operational impact analysis.
- Granularity-aware joins help prevent **duplicate records and incorrect metric aggregation**.

# Proposed Star Schema

In [27]:
schema = pd.DataFrame({

    "Table":[
        "Fact_Order_Items",
        "Dim_Date",
        "Dim_Product",
        "Dim_Customer",
        "Dim_Department",
        "Dim_Category",
        "Dim_Market",
        "Dim_Country",
        "Dim_Commodity",
        "Dim_Fuel",
        "Dim_ExchangeRate",
        "Dim_Weather",
        "Dim_Holiday"
    ],

    "Table Type":[
        "Fact",
        "Dimension",
        "Dimension",
        "Dimension",
        "Dimension",
        "Dimension",
        "Dimension",
        "Dimension",
        "Dimension",
        "Dimension",
        "Dimension",
        "Dimension",
        "Dimension"
    ]

})

schema

,Table,Table Type
0,Fact_Order_Items,Fact
1,Dim_Date,Dimension
2,Dim_Product,Dimension
3,Dim_Customer,Dimension
4,Dim_Department,Dimension
5,Dim_Category,Dimension
6,Dim_Market,Dimension
7,Dim_Country,Dimension
8,Dim_Commodity,Dimension
9,Dim_Fuel,Dimension


### Observations

- The proposed warehouse follows a **star schema** with `Fact_Order_Items` as the central fact table.
- The model contains **1 fact table and 12 dimension tables**.
- Dimensions provide descriptive and contextual attributes for **transaction, customer, product, geographic, and external-factor analysis**.
- The design supports efficient **SQL querying, Power BI reporting, KPI analysis, and predictive modeling**.
- The schema will be refined during physical implementation based on **data availability and validated relationships**.

# Enterprise KPI Layer

The warehouse will support the following KPIs.

• Procurement Cost

• Vendor Reliability

• Lead Time

• Inventory Turnover

• Procurement Savings

• Supplier Risk

• On-Time Delivery %

• Fuel Cost Impact

• Commodity Cost Impact

• Currency Impact

• Weather Delay %

• Purchase Forecast Accuracy

# Next Steps

The enterprise data model has now been defined.

The next notebook focuses on transforming the raw datasets into analysis-ready tables through:

- Missing value treatment
- Duplicate handling
- Data type corrections
- Standardization
- Data validation

This cleaned data will serve as the foundation for feature engineering and predictive analytics.